# LRN Training Example

This notebook trains an `LRNModel` on multilayer SIMNRA datasets discovered under a user-provided root folder.

The workflow is:

1. Recursively scan an absolute dataset root for `.h5`, `.hdf5`, or `.hf5` files.
2. Pad each case into the open-parameter layout of the max-layer dataset.
3. Build an LRN schema from that max-layer input spec.
4. Train the network on one selected output method such as `RBS`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys 
sys.path.append("../")

import matplotlib.pyplot as plt
import numpy as np
import torch

from ibamlkit.data import DatasetBatchReader
from ibamlkit.models.forward import LRNModel, LTNModel, build_lrn_model_schema
from ibamlkit.pileup import (
    apply_channel_space_pileup,
    convert_energy_spectra_to_channel_space_and_pileup,
    convert_to_channel_space_and_pileup_batch,
    fast_pileup_batch,
    rebin_spectra_to_energy_space,
    rebin_histogram,
    resolve_channel_conversion_arrays,
)
from ibamlkit.training import (
    ConstantFactorTransform,
    EpochSchedule,
    IdentityTransform,
    Chi2Loss,
    MinMaxScaler,
    PeakAwareLoss,
    SupervisedTrainer,
    TransformPipeline,
    prepare_variable_layer_surrogate_dataset,
    shuffle_in_unison,
    split_train_val_test,
)
from ibamlkit.validation import (
    SIMNRABatchSimulator,
    SimulationBatchResult,
    SurrogateBatchSimulator,
    calculate_chi2_batch,
    fit_open_parameters,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
DATASET_FILE_SUFFIXES = {".h5", ".hdf5", ".hf5"}


def require_absolute_directory(path: Path) -> Path:
    path = Path(path)
    if not path.is_absolute():
        raise ValueError(f"Dataset scan root must be an absolute path, got: {path}")
    if not path.exists():
        raise FileNotFoundError(f"Dataset scan root does not exist: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"Dataset scan root is not a directory: {path}")
    return path


def collect_dataset_file_groups(scan_root: Path) -> list[tuple[Path, list[Path]]]:
    scan_root = require_absolute_directory(scan_root)
    grouped_paths: dict[Path, list[Path]] = {}
    for path in sorted(scan_root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in DATASET_FILE_SUFFIXES:
            continue
        grouped_paths.setdefault(path.parent, []).append(path)
    if not grouped_paths:
        raise FileNotFoundError(
            f"No dataset files with suffixes {sorted(DATASET_FILE_SUFFIXES)} were found under {scan_root}"
        )
    return sorted(grouped_paths.items(), key=lambda item: str(item[0]).lower())


def load_case_dataset(paths: list[Path]):
    reader = DatasetBatchReader()
    dataset = reader.load_many(paths)
    return dataset, paths


def print_array_stats(name: str, x: np.ndarray) -> None:
    x = np.asarray(x, dtype=np.float64)
    print(
        f"{name}: shape={x.shape}, min={x.min():.6g}, max={x.max():.6g}, "
        f"mean={x.mean():.6g}, median={np.median(x):.6g}"
    )


In [ ]:
dataset_root = Path(r"D:/Developments/IBAMLKit/examples/datasets")
method_name = "RBS"
target_width = 4000
target_scale_factor = 1e-6
energy_bin_width = 1.0
energy_spectrum_scale = 1e-3
seed = 7
val_count = 2000
test_count = 2000

dataset_root = require_absolute_directory(dataset_root)
case_groups = collect_dataset_file_groups(dataset_root)
datasets = []
for case_dir, paths in case_groups:
    try:
        dataset, paths = load_case_dataset(paths)
        n_layers = int(dataset.input_spec.generation_info.get("n_layers", 0))
        print(
            f"Loaded case {case_dir}: {len(paths)} file(s), "
            f"samples={dataset.sample_count}, n_layers={n_layers}"
        )
        datasets.append(dataset)
    except Exception as e:
        print(f"Error loading case {case_dir}: {e}")

if not datasets:
    raise RuntimeError(f"No datasets could be loaded from {dataset_root}")

reference_dataset = max(
    datasets,
    key=lambda dataset: int(dataset.input_spec.generation_info.get("n_layers", 0)),
)
bootstrap_schema = build_lrn_model_schema(
    reference_dataset.input_spec,
    model_name=f"lrn_{method_name.lower()}",
    task_method_names=[method_name],
    output_spectra_lengths={method_name: 1},
)
prepared = prepare_variable_layer_surrogate_dataset(
    datasets,
    schema=bootstrap_schema,
    method_name=method_name,
)
reference_dataset = prepared.reference_dataset
reference_open_parameter_names = [parameter.name for parameter in reference_dataset.input_spec.open_parameters]
full_open_parameter_values = np.asarray(prepared.inputs_full, dtype=np.float32)
channel_targets_raw = np.asarray(prepared.targets, dtype=np.float32)
channel_target_lengths = prepared.target_lengths

calibration_offset_all, calibration_linear_all, calibration_quadratic_all, normalization_factor_all = resolve_channel_conversion_arrays(
    reference_dataset.input_spec,
    reference_open_parameter_names,
    full_open_parameter_values,
    method_name=method_name,
    energy_spectrum_scale=energy_spectrum_scale,
)

energy_targets_raw, energy_target_lengths, energy_bin_edges = rebin_spectra_to_energy_space(
    channel_targets_raw,
    channel_target_lengths,
    calibration_offset=calibration_offset_all,
    calibration_linear=calibration_linear_all,
    calibration_quadratic=calibration_quadratic_all,
    energy_bin_width=energy_bin_width,
    target_width=target_width,
    energy_spectrum_scale=energy_spectrum_scale,
)

schema = build_lrn_model_schema(
    reference_dataset.input_spec,
    model_name=f"lrn_{method_name.lower()}_e_space",
    task_method_names=[method_name],
    output_spectra_lengths={method_name: int(energy_targets_raw.shape[1])},
)
x = prepared.inputs_selected
y_raw = energy_targets_raw
y_lengths = energy_target_lengths

output_transform_steps = [IdentityTransform()]
if target_scale_factor is not None:
    output_transform_steps.append(ConstantFactorTransform(target_scale_factor))
output_transform = TransformPipeline(output_transform_steps)
y = output_transform.fit_transform(y_raw)

print_array_stats("Raw channel-space targets", channel_targets_raw)
print_array_stats("Raw E-space targets", y_raw)
print_array_stats("Transformed E-space targets", y)

print("Dataset root:", dataset_root)
print("Reference layer count:", reference_dataset.input_spec.generation_info.get("n_layers"))
print("Input matrix shape:", x.shape)
print("E-space output matrix shape:", y.shape)
print("Energy bin width:", energy_bin_width)
print("Energy bin edge count:", energy_bin_edges.shape[0])
print("Schema input dimension:", schema.inputs.dimension)
print("Schema E-space output width:", schema.outputs.spectra_lengths[method_name])

In [ ]:
plt.hist(dataset.open_parameter_values[:,0 ])

In [ ]:
sample_count = x.shape[0]
permutation = np.random.default_rng(seed).permutation(sample_count)
x = np.asarray(x[permutation], dtype=np.float32)
y = np.asarray(y[permutation], dtype=np.float32)
y_lengths = None if y_lengths is None else np.asarray(y_lengths[permutation], dtype=np.int32)
channel_targets_raw = np.asarray(channel_targets_raw[permutation], dtype=np.float32)
channel_target_lengths = None if channel_target_lengths is None else np.asarray(channel_target_lengths[permutation], dtype=np.int32)
full_open_parameter_values = np.asarray(full_open_parameter_values[permutation], dtype=np.float32)

split = split_train_val_test(
    x,
    y,
    val_count=val_count,
    test_count=test_count,
    target_lengths=y_lengths,
)

x_train = split.train_inputs
y_train = split.train_targets
x_val = split.val_inputs
y_val = split.val_targets
x_test = split.test_inputs
y_test = split.test_targets
test_lengths = split.test_target_lengths

train_count = x_train.shape[0]
val_end = train_count + x_val.shape[0]
full_open_parameter_values_train = full_open_parameter_values[:train_count]
full_open_parameter_values_val = full_open_parameter_values[train_count:val_end]
full_open_parameter_values_test = full_open_parameter_values[val_end:]
channel_targets_test = channel_targets_raw[val_end:]
channel_target_lengths_test = None if channel_target_lengths is None else channel_target_lengths[val_end:]

input_scaler = MinMaxScaler(low=0.0, high=1.0)
x_train_scaled = input_scaler.fit_transform(x_train)
x_val_scaled = input_scaler.transform(x_val)
x_test_scaled = input_scaler.transform(x_test)

print("Train:", x_train_scaled.shape, y_train.shape)
print("Val:", x_val_scaled.shape, y_val.shape)
print("Test:", x_test_scaled.shape, y_test.shape)
print("Channel-space test targets:", channel_targets_test.shape)

In [ ]:
test_lengths 

In [ ]:
for i in range(3) : 
    plt.plot(y_train[i], label="Train")


In [ ]:

model = LTNModel(
    schema,
    model_dim=256,
    num_heads=1,
    num_encoder_layers=2,
    feedforward_dim=768,
    dropout=0.1,
    decoder_hidden_sizes=(512, 512),
    refiner_hidden_channels=32,
    refiner_kernel_size=17,
)

modell = LRNModel(
    schema,
    hidden_size= 256,
    contribution_size= 256,
    setup_embedding_dim = 32,
    layer_embedding_dim = 256,
    block_hidden_sizes = (512, 512),
    decoder_hidden_sizes = (768, 768),
    refiner_hidden_channels=32,
    refiner_kernel_size=17,
)


trainer = SupervisedTrainer(
    device=device,
    loss_fn=PeakAwareLoss(
        amplitude_weight=1.0,
        gradient_weight=0.25,
        curvature_weight=0.05,
    ),
    optimizer_name="adamw",
    weight_decay=1e-3,
    max_grad_norm=1.0,
    early_stopping_patience=10,
    verbose=True,
    log_every_epochs=1,
)

result = trainer.fit(
    model,
    train_inputs=x_train_scaled,
    train_targets=y_train,
    val_inputs=x_val_scaled,
    val_targets=y_val,
    schedule=[
        #EpochSchedule(learning_rate=5e-3, epochs=10, batch_size=256),
        EpochSchedule(learning_rate=1e-3, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=1e-4, epochs=20, batch_size=1024),
        EpochSchedule(learning_rate=5e-5, epochs=5, batch_size=1024),
    ],
)

print(result)

In [ ]:
pred_test_transformed = model.predict(x_test_scaled).cpu().numpy()
pred_test_e = output_transform.inverse_transform(pred_test_transformed)
y_test_e = output_transform.inverse_transform(y_test)

print_array_stats("Predicted E-space test spectra", pred_test_e)
chi2_e = np.mean((pred_test_e - y_test_e) ** 2 / (y_test_e + 1.0), axis=1)
print("E-space test mean chi2:", float(np.mean(chi2_e)))
print("E-space test median chi2:", float(np.median(chi2_e)))

real_time = 8.290e6
live_time = 7.930e6
fudge_factor = 0.4500
real_times_test = np.full((x_test.shape[0],), real_time, dtype=np.float64)
live_times_test = np.full((x_test.shape[0],), live_time, dtype=np.float64)
fudge_factors_test = np.full((x_test.shape[0],), fudge_factor, dtype=np.float64)

calibration_offset_test, calibration_linear_test, calibration_quadratic_test, normalization_factor_test = resolve_channel_conversion_arrays(
    reference_dataset.input_spec,
    reference_open_parameter_names,
    full_open_parameter_values_test,
    method_name=method_name,
    energy_spectrum_scale=energy_spectrum_scale,
)

pred_test_channel_pileup = convert_energy_spectra_to_channel_space_and_pileup(
    pred_test_e,
    calibration_offset=calibration_offset_test,
    calibration_linear=calibration_linear_test,
    calibration_quadratic=calibration_quadratic_test,
    real_times=real_times_test,
    live_times=live_times_test,
    fudge_factors=fudge_factors_test,
    normalization_factors=normalization_factor_test,
)
y_test_channel_pileup = convert_energy_spectra_to_channel_space_and_pileup(
    y_test_e,
    calibration_offset=calibration_offset_test,
    calibration_linear=calibration_linear_test,
    calibration_quadratic=calibration_quadratic_test,
    real_times=real_times_test,
    live_times=live_times_test,
    fudge_factors=fudge_factors_test,
    normalization_factors=normalization_factor_test,
)
chi2_channel_pileup = np.mean(
    (pred_test_channel_pileup - y_test_channel_pileup) ** 2 / (y_test_channel_pileup + 1.0),
    axis=1,
)
print("Channel-space + pileup test mean chi2:", float(np.mean(chi2_channel_pileup)))
print("Channel-space + pileup test median chi2:", float(np.median(chi2_channel_pileup)))

n_plot = min(20, x_test.shape[0])
fig, axes = plt.subplots(n_plot, 2, figsize=(12, 2.5 * n_plot), sharex=False)
if n_plot == 1:
    axes = np.asarray([axes])

for row_index in range(n_plot):
    e_length = y_test_e.shape[1] if test_lengths is None else min(int(test_lengths[row_index]), y_test_e.shape[1])
    axes[row_index, 0].plot(y_test_e[row_index, :e_length], label="target")
    axes[row_index, 0].plot(pred_test_e[row_index, :e_length], linestyle="--", label="prediction")
    axes[row_index, 0].set_ylabel(f"sample {row_index}")
    axes[row_index, 0].set_title("E-space")
    axes[row_index, 1].plot(y_test_channel_pileup[row_index], label="target")
    axes[row_index, 1].plot(pred_test_channel_pileup[row_index], linestyle="--", label="prediction")
    axes[row_index, 1].set_title("Channel + pileup")
    if row_index == 0:
        axes[row_index, 0].legend()
        axes[row_index, 1].legend()

axes[-1, 0].set_xlabel("energy bin")
axes[-1, 1].set_xlabel("channel")
plt.tight_layout()
plt.show()

In [ ]:
artifact_dir = Path.cwd() / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
artifact_path = artifact_dir / f"lrn_{method_name.lower()}_state_dict.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "schema": schema,
        "input_scaler": input_scaler,
        "output_transform": output_transform,
        "training_result": result,
    },
    artifact_path,
)
print("Saved:", artifact_path)

## Parameter Fitting On Test Spectra

The cells below fit the open parameters for a small subset of the test spectra using both the trained surrogate model and SIMNRA.

Notes:
- Surrogate fitting ignores `nthreads` because model inference is already vectorized.
- SIMNRA fitting can be substantially slower; keep `fit_sample_count` small unless you want a long run.


In [ ]:
fit_sample_count = 4
simnra_fit_threads = 4
surrogate_fit_algo = "L_BFGS_B"
simnra_fit_algo = "ARRDE"
surrogate_fit_maxevals = 800
simnra_fit_maxevals = 3000
fit_seed = 123

fit_sample_count = min(fit_sample_count, x_test.shape[0])
x_fit_true = np.asarray(x_test[:fit_sample_count], dtype=np.float32)
observed_channel_pileup = np.asarray(y_test_channel_pileup[:fit_sample_count], dtype=np.float32)
observed_spectra = {method_name: observed_channel_pileup}
observed_lengths = {method_name: np.full((fit_sample_count,), observed_channel_pileup.shape[1], dtype=np.int32)}

open_parameters = list(reference_dataset.input_spec.open_parameters)
lower_bounds = np.asarray([
    0.0 if parameter.lower_bound is None else parameter.lower_bound
    for parameter in open_parameters
], dtype=np.float32)
upper_bounds = np.asarray([
    1.0 if parameter.upper_bound is None else parameter.upper_bound
    for parameter in open_parameters
], dtype=np.float32)
rng = np.random.default_rng(fit_seed)
initial_guess = lower_bounds + rng.uniform(size=x_fit_true.shape).astype(np.float32) * (upper_bounds - lower_bounds)

class SurrogateChannelPileupSimulator:
    def __init__(self, input_spec, schema, model, input_transform, output_inverse_transform, method_name, real_time, live_time, fudge_factor, energy_spectrum_scale):
        self.input_spec = input_spec
        self.schema = schema
        self.model = model
        self.input_transform = input_transform
        self.output_inverse_transform = output_inverse_transform
        self.method_name = method_name
        self.real_time = float(real_time)
        self.live_time = float(live_time)
        self.fudge_factor = float(fudge_factor)
        self.energy_spectrum_scale = float(energy_spectrum_scale)
        self.open_parameter_names = [parameter.name for parameter in input_spec.open_parameters]
        self.base = SurrogateBatchSimulator(
            input_spec=input_spec,
            schema=schema,
            model=model,
            input_transform=input_transform,
            output_inverse_transform=output_inverse_transform,
        )
        self.method_names = [method_name]

    def simulate_batch(self, open_parameter_values, *, fixed_parameter_overrides=None, nthreads=1):
        del nthreads
        energy_result = self.base.simulate_batch(
            open_parameter_values,
            fixed_parameter_overrides=fixed_parameter_overrides,
        )
        a, b, c, r = resolve_channel_conversion_arrays(
            self.input_spec,
            self.open_parameter_names,
            np.asarray(open_parameter_values, dtype=np.float32),
            method_name=self.method_name,
            energy_spectrum_scale=self.energy_spectrum_scale,
        )
        sample_count = np.asarray(open_parameter_values).shape[0]
        spectra = convert_energy_spectra_to_channel_space_and_pileup(
            energy_result.spectra[self.method_name],
            calibration_offset=a,
            calibration_linear=b,
            calibration_quadratic=c,
            real_times=np.full((sample_count,), self.real_time, dtype=np.float64),
            live_times=np.full((sample_count,), self.live_time, dtype=np.float64),
            fudge_factors=np.full((sample_count,), self.fudge_factor, dtype=np.float64),
            normalization_factors=r,
        )
        lengths = np.full((sample_count,), spectra.shape[1], dtype=np.int32)
        return SimulationBatchResult(spectra={self.method_name: spectra}, spectra_lengths={self.method_name: lengths})

    def close(self):
        close = getattr(self.base, "close", None)
        if callable(close):
            close()


class SIMNRAChannelPileupSimulator:
    def __init__(self, input_spec, method_name, real_time, live_time, fudge_factor):
        self.input_spec = input_spec
        self.method_name = method_name
        self.real_time = float(real_time)
        self.live_time = float(live_time)
        self.fudge_factor = float(fudge_factor)
        self.base = SIMNRABatchSimulator(input_spec)
        self.method_names = [method_name]

    def simulate_batch(self, open_parameter_values, *, fixed_parameter_overrides=None, nthreads=1):
        channel_result = self.base.simulate_batch(
            open_parameter_values,
            fixed_parameter_overrides=fixed_parameter_overrides,
            nthreads=nthreads,
        )
        sample_count = np.asarray(open_parameter_values).shape[0]
        spectra = apply_channel_space_pileup(
            channel_result.spectra[self.method_name],
            real_times=np.full((sample_count,), self.real_time, dtype=np.float64),
            live_times=np.full((sample_count,), self.live_time, dtype=np.float64),
            fudge_factors=np.full((sample_count,), self.fudge_factor, dtype=np.float64),
        )
        lengths = np.full((sample_count,), spectra.shape[1], dtype=np.int32)
        return SimulationBatchResult(spectra={self.method_name: spectra}, spectra_lengths={self.method_name: lengths})

    def close(self):
        close = getattr(self.base, "close", None)
        if callable(close):
            close()

print("Fitting sample count:", fit_sample_count)
print("Initial guess shape:", initial_guess.shape)
print("Observed channel+pileup spectra shape:", observed_channel_pileup.shape)

In [ ]:
surrogate_simulator = SurrogateChannelPileupSimulator(
    input_spec=reference_dataset.input_spec,
    schema=schema,
    model=model,
    input_transform=input_scaler,
    output_inverse_transform=output_transform,
    method_name=method_name,
    real_time=real_time,
    live_time=live_time,
    fudge_factor=fudge_factor,
    energy_spectrum_scale=energy_spectrum_scale,
)

surrogate_fit_result = fit_open_parameters(
    surrogate_simulator,
    reference_dataset.input_spec,
    observed_spectra,
    observed_lengths=observed_lengths,
    initial_open_parameter_values=initial_guess,
    algo=surrogate_fit_algo,
    maxevals=surrogate_fit_maxevals,
    rel_tol=0.0,
    seed=fit_seed,
)
surrogate_fit_params = surrogate_fit_result.best_open_parameter_values
surrogate_eval_simulator = SurrogateChannelPileupSimulator(
    input_spec=reference_dataset.input_spec,
    schema=schema,
    model=model,
    input_transform=input_scaler,
    output_inverse_transform=output_transform,
    method_name=method_name,
    real_time=real_time,
    live_time=live_time,
    fudge_factor=fudge_factor,
    energy_spectrum_scale=energy_spectrum_scale,
)
surrogate_fit_chi2 = calculate_chi2_batch(
    surrogate_eval_simulator,
    surrogate_fit_params,
    observed_spectra,
    observed_lengths=observed_lengths,
).total

relative_param_error = np.abs(surrogate_fit_params - x_fit_true) / np.maximum(np.abs(x_fit_true), 1e-12)
print("Surrogate fit mean chi2:", float(np.mean(surrogate_fit_chi2)))
print("Surrogate fit median chi2:", float(np.median(surrogate_fit_chi2)))
print("Surrogate fit mean relative parameter error:", float(np.mean(relative_param_error)))
print("Surrogate fit results:")
for sample in surrogate_fit_result.samples:
    print(sample)

In [ ]:
try:
    simnra_simulator = SIMNRAChannelPileupSimulator(
        reference_dataset.input_spec,
        method_name=method_name,
        real_time=real_time,
        live_time=live_time,
        fudge_factor=fudge_factor,
    )
    simnra_fit_result = fit_open_parameters(
        simnra_simulator,
        reference_dataset.input_spec,
        observed_spectra,
        observed_lengths=observed_lengths,
        initial_open_parameter_values=initial_guess,
        algo=simnra_fit_algo,
        maxevals=simnra_fit_maxevals,
        rel_tol=0.0,
        seed=fit_seed,
        nthreads=simnra_fit_threads,
    )
    simnra_fit_params = simnra_fit_result.best_open_parameter_values
    simnra_eval_simulator = SIMNRAChannelPileupSimulator(
        reference_dataset.input_spec,
        method_name=method_name,
        real_time=real_time,
        live_time=live_time,
        fudge_factor=fudge_factor,
    )
    simnra_fit_chi2 = calculate_chi2_batch(
        simnra_eval_simulator,
        simnra_fit_params,
        observed_spectra,
        observed_lengths=observed_lengths,
        nthreads=simnra_fit_threads,
    ).total
    simnra_eval_simulator.close()
    simnra_relative_param_error = np.abs(simnra_fit_params - x_fit_true) / np.maximum(np.abs(x_fit_true), 1e-12)
    print("SIMNRA fit mean chi2:", float(np.mean(simnra_fit_chi2)))
    print("SIMNRA fit median chi2:", float(np.median(simnra_fit_chi2)))
    print("SIMNRA fit mean relative parameter error:", float(np.mean(simnra_relative_param_error)))
    print("SIMNRA fit results:")
    for sample in simnra_fit_result.samples:
        print(sample)
except Exception as exc:
    print("SIMNRA fitting skipped or failed:", exc)